# Cultural Heritage Social Media & AI Dataset Analysis

This notebook performs comprehensive analysis on the dataset including:
1. Frequency analysis of platforms, modalities, tasks, and models
2. Numeric conversion of dataset size metrics
3. Date range parsing and duration calculation
4. Cross-dataset comparison

In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from calendar import monthrange
import warnings
warnings.filterwarnings('ignore')

try:
    import inflect
    _INFLECT = inflect.engine()
except ImportError:
    _INFLECT = None

from src.term_mapping.mapping_dictionaries import (
    get_platform_mappings,
    get_modality_mappings,
    get_model_mappings,
    get_task_mappings
)

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 100)

# Set style for visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## Part 1: Data Loading and Preprocessing Functions

In [ ]:
def parse_semicolon_field(field_value):
    """
    Parse semicolon-separated fields and remove parentheses with counts.
    Example: 'Twitter (5); Facebook (3)' -> ['Twitter', 'Facebook']
    """
    if pd.isna(field_value) or field_value == '':
        return []
    
    # Split by semicolon
    items = [item.strip() for item in str(field_value).split(';')]
    
    # Remove parentheses and their contents
    cleaned_items = []
    for item in items:
        # Remove anything in parentheses
        cleaned = re.sub(r'\s*\([^)]*\)', '', item).strip()
        if cleaned:
            cleaned_items.append(cleaned)
    
    return cleaned_items


In [ ]:
def normalize_term(term):
    """
    Normalize terms by converting to lowercase and handling common variations.
    """

    if pd.isna(term) or term == '':
        return ''

    term = str(term).lower().strip()

    # Remove common variations
    term = re.sub(r'[^a-z0-9\s]', '', term)
    term = re.sub(r'\s+', ' ', term).strip()

    return term


def create_frequency_dict(df, column_name, normalize=True, mapping_type=None):
    """
    Create a frequency dictionary from a semicolon-separated column.
    """
    all_items = []

    for value in df[column_name].dropna():
        items = parse_semicolon_field(value)
        if normalize:
            if mapping_type is None:
                items = list(set([normalize_term(item) for item in items]))
            else:
                #print('Mapping {}'.format(types))
                if mapping_type == 'modality':
                    mappings = get_modality_mappings()
                elif mapping_type == 'platform':
                    mappings = get_platform_mappings()
                elif mapping_type == 'model':
                    mappings = get_model_mappings()
                elif mapping_type == 'task':
                    mappings = get_task_mappings()
                else:
                    # Try all mappings
                    mappings = {
                        **get_modality_mappings(),
                        **get_platform_mappings(),
                        **get_model_mappings(),
                        **get_task_mappings()
                    }
                items = list(set([mappings[normalize_term(item)] if normalize_term(item) in mappings else normalize_term(item) for item in items]))
        all_items.extend(items)

    # Count frequencies
    freq_dict = dict(Counter(all_items))

    # Sort by frequency (descending)
    freq_dict = dict(sorted(freq_dict.items(), key=lambda x: x[1], reverse=True))

    return freq_dict


print("✓ Data preprocessing functions loaded successfully")

In [ ]:
def singularize_token(tok: str) -> str:
    tok = tok.strip()
    if not tok:
        return tok

    # Don't touch short acronyms much (llms -> llm is okay, but keep "asr", "tts")
    if len(tok) <= 2:
        return tok

    if _INFLECT is not None:
        s = _INFLECT.singular_noun(tok)
        return s if s else tok

    # Fallback heuristic (if inflect isn't installed)
    if tok.endswith("ies") and len(tok) > 3:
        return tok[:-3] + "y"
    if tok.endswith(("ches", "shes", "xes", "ses", "zes")) and len(tok) > 3:
        return tok[:-2]  # remove "es"
    if tok.endswith("s") and not tok.endswith(("ss", "us", "is")) and len(tok) > 3:
        return tok[:-1]
    return tok


def singularize_phrase(phrase: str) -> str:
    # preserve underscores vs spaces
    sep = "_" if "_" in phrase else " "
    parts = phrase.split(sep)
    return sep.join(singularize_token(p) for p in parts)

def normalize_term(term: str, mapping_type: str = 'general') -> str:
    """
    Normalize a single term using the appropriate mapping.
    Applies mappings iteratively until no more changes occur.

    Args:
        term: The term to normalize
        mapping_type: Type of mapping ('modality', 'platform', 'model', 'task', or 'general')

    Returns:
        Normalized term
    """
    if pd.isna(term) or term == '':
        return ''

    term = str(term).lower().strip()

    # Remove extra spaces and normalize
    term = re.sub(r'\s+', ' ', term)

    # Get appropriate mapping
    if mapping_type == 'modality':
        mappings = get_modality_mappings()
    elif mapping_type == 'platform':
        mappings = get_platform_mappings()
    elif mapping_type == 'model':
        mappings = get_model_mappings()
    elif mapping_type == 'task':
        mappings = get_task_mappings()
    else:
        # No mapping
        mappings = {}

    # ITERATIVE NORMALIZATION - Apply mappings until no more changes
    max_iterations = 5  # Prevent infinite loops
    for _ in range(max_iterations):
        original_term = term

        # Direct match first
        if term in mappings:
            term = mappings[term]
            if term == original_term:  # No change, stop
                break
            continue

        # Try with underscores instead of spaces
        underscore_term = term.replace(' ', '_')
        if underscore_term in mappings:
            term = mappings[underscore_term]
            if term == original_term:
                break
            continue

        # Try with cleaned term (remove special characters except spaces)
        cleaned_term = re.sub(r'[^a-z0-9\s]', '', term)
        if cleaned_term in mappings:
            term = mappings[cleaned_term]
            if term == original_term:
                break
            continue

        # Try cleaned term with underscores
        cleaned_underscore = cleaned_term.replace(' ', '_')
        if cleaned_underscore in mappings:
            term = mappings[cleaned_underscore]
            if term == original_term:
                break
            continue

        # Partial match (for terms within longer phrases)
        # Only match if the key is a complete word (surrounded by word boundaries)
        # Sort by length (longest first) to match more specific terms first
        matched = False
        for key, value in sorted(mappings.items(), key=lambda x: len(x[0]), reverse=True):
            # Match only if key is a complete word (with word boundaries)
            # Use regex to check if key appears as complete word
            import re as re_inner
            # Check term
            if re_inner.search(r'\b' + re_inner.escape(key) + r'\b', term):
                term = value
                matched = True
                break
            # Check cleaned_term
            if re_inner.search(r'\b' + re_inner.escape(key) + r'\b', cleaned_term):
                term = value
                matched = True
                break
            # Check underscore_term
            if re_inner.search(r'\b' + re_inner.escape(key) + r'\b', underscore_term):
                term = value
                matched = True
                break

        if not matched or term == original_term:
            # No more changes possible, return underscore version
            return singularize_phrase(underscore_term) if underscore_term else singularize_phrase(term)

    # Return final normalized term
    return singularize_phrase(term)


def parse_semicolon_field(field_value: str, mapping_type: str = 'general') -> List[str]:
    """
    Parse semicolon-separated fields, remove parentheses, and normalize.

    Args:
        field_value: The field value to parse
        mapping_type: Type of normalization to apply

    Returns:
        List of normalized terms
    """
    if pd.isna(field_value) or field_value == '':
        return []

    # Split by semicolon
    items = [item.strip() for item in str(field_value).split(';')]

    # Remove parentheses and normalize
    normalized_items = []
    for item in items:
        # Remove anything in parentheses
        cleaned = re.sub(r'\s*\([^)]*\)', '', item).strip()
        if cleaned:
            normalized = normalize_term(cleaned, mapping_type)
            if normalized:
                normalized_items.append(normalized)

    return list(set(normalized_items))


def get_normalized_frequencies(df: pd.DataFrame, column_name: str,
                               mapping_type: str = 'general') -> Dict[str, int]:
    """
    Create a frequency dictionary from a column with normalization.

    Args:
        df: DataFrame containing the data
        column_name: Name of the column to analyze
        mapping_type: Type of normalization ('modality', 'platform', 'model', 'task', or 'general')

    Returns:
        Dictionary of {term: frequency} sorted by frequency
    """
    all_items = []

    for value in df[column_name].dropna():
        items = parse_semicolon_field(value, mapping_type)
        all_items.extend(items)

    # Count frequencies
    freq_dict = Counter(all_items)
    freq_dict = dict(sorted(freq_dict.items(), key=lambda x: x[1], reverse=True))

    return dict(freq_dict)

In [ ]:
def get_top_n(frequency_dict: Dict[str, int], n: int = 10) -> Dict[str, int]:
    """
    Get the top N items from a frequency dictionary.

    Args:
        frequency_dict: Dictionary of term frequencies
        n: Number of top items to return

    Returns:
        Dictionary with top N items
    """
    sorted_items = sorted(frequency_dict.items(), key=lambda x: x[1], reverse=True)
    return dict(sorted_items[:n])


def print_frequency_summary(frequency_dict: Dict[str, int],
                           category_name: str = "Items",
                           top_n: int = 20):
    """
    Print a formatted summary of frequencies.

    Args:
        frequency_dict: Dictionary of term frequencies
        category_name: Name of the category (for display)
        top_n: Number of top items to show
    """
    total = sum(frequency_dict.values())
    unique = len(frequency_dict)

    print(f"\n{category_name} Analysis:")
    print(f"  Total occurrences: {total}")
    print(f"  Unique categories: {unique}")
    print(f"\nTop {top_n} {category_name}:")
    print("-" * 60)

    sorted_items = sorted(frequency_dict.items(), key=lambda x: x[1], reverse=True)
    for i, (term, count) in enumerate(sorted_items[:top_n], 1):
        percentage = (count / total) * 100
        print(f"{i:2d}. {term:40s} {count:4d} ({percentage:5.1f}%)")


# ============================================================================
# CONVENIENCE FUNCTIONS
# ============================================================================

def analyze_dataset(df: pd.DataFrame,
                   columns: Dict[str, str] = None) -> Dict[str, Dict[str, int]]:
    """
    Analyze multiple columns in a dataset and return normalized frequencies.

    Args:
        df: DataFrame to analyze
        columns: Dictionary mapping column names to mapping types
                 e.g., {'platforms_detected': 'platform', 'task_types': 'task'}

    Returns:
        Dictionary mapping column names to frequency dictionaries
    """
    if columns is None:
        columns = {
            'platforms_detected': 'platform',
            'data_modalities_detected': 'modality',
            'task_types': 'task',
            'model_families': 'model',
            'specific_models_mentioned': 'model',
            'all_models': 'model',
            'heritage_terms_detected': '',
            'heritage_management_actions': '',
        }

    results = {}
    for col_name, mapping_type in columns.items():
        if col_name in df.columns:
            results[col_name] = get_normalized_frequencies(df, col_name, mapping_type)

    return results

def export_frequencies_to_json(frequencies: Dict[str, Dict[str, int]],
                               output_path: str):
    """
    Export frequency dictionaries to JSON file.

    Args:
        frequencies: Dictionary of frequency dictionaries
        output_path: Path to save JSON file
    """
    import json

    # Sort each frequency dict by count
    sorted_frequencies = {}
    for category, freq_dict in frequencies.items():
        sorted_frequencies[category] = dict(
            sorted(freq_dict.items(), key=lambda x: x[1], reverse=True)
        )

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(sorted_frequencies, f, indent=2, ensure_ascii=False)

    print(f"✓ Frequencies exported to: {output_path}")

In [ ]:
def get_frequent_terms(frequency_dict: Dict[str, int],
                    category_name: str = "platform",
                    min_frequency: int = 10) -> List[str]:
    """
    Get terms that appear at least min_frequency times.

    Args:
        df: DataFrame with the data
        column_name: Column to analyze
        mapping_type: Type of mapping ('platform', 'modality', 'task', 'model')
        min_frequency: Minimum number of occurrences

    Returns:
        List of frequent terms
    """
    freq_dict = frequency_dict[category_name]
    frequent_terms = [term for term, count in freq_dict.items() if count >= min_frequency]
    return frequent_terms

In [ ]:
def create_binary_columns(df: pd.DataFrame,
                         frequency_dict: Dict[str, int],
                         column_name: str,
                         min_frequency: int = 10,
                         prefix: [str] = None,
                         categories: [Dict[str, str]] = None) -> pd.DataFrame:
    """
    Create binary columns for each frequent term.

    For each term appearing >= min_frequency times, creates a column
    indicating whether that term appears in each row.

    Args:
        df: DataFrame with the data
        column_name: Column to analyze (e.g., 'task_types', 'model_families')
        mapping_type: Type of mapping ('platform', 'modality', 'task', 'model')
        min_frequency: Minimum occurrences to create column (default: 10)
        prefix: Prefix for new columns (default: uses column_name)

    Returns:
        DataFrame with original data plus binary columns

    Example:
        df_expanded = create_binary_columns(df, 'task_types', 'task', min_frequency=10)
        # Creates columns like: task_classification, task_clustering, etc.
    """
    # Get frequent terms
    if categories is None:
        categories = {
            'platforms_detected': 'platform',
            'data_modalities_detected': 'modality',
            'task_types': 'task',
            'model_families': 'model',
            'specific_models_mentioned': 'model',
            'all_models': 'model',
            'heritage_terms_detected': '',
            'heritage_management_actions': '',
        }
    mapping_type = categories[column_name]
    frequent_terms = get_frequent_terms(frequency_dict, column_name, min_frequency)

    # Determine prefix
    if prefix is None:
        prefix = mapping_type

    # Create a copy of the dataframe
    df_result = df.copy()

    print(f"Creating binary columns for {len(frequent_terms)} frequent terms from '{column_name}'...")
    print(f"(Terms appearing >= {min_frequency} times)")

    # For each frequent term, create a binary column
    for term in frequent_terms:
        col_name = f"{prefix}_{term}"

        # Check each row for this term
        df_result[col_name] = df[column_name].apply(
            lambda x: int(term in parse_semicolon_field(str(x), mapping_type))
            if pd.notna(x) else 0
        )

    print(f"✓ Created {len(frequent_terms)} binary columns")
    print(f"\nExample columns: {', '.join([f'{prefix}_{t}' for t in frequent_terms[:5]])}")

    return df_result

In [ ]:
def create_all_binary_columns(df: pd.DataFrame,
                              frequency_dict: Dict[str, int],
                              min_frequency: int = 10,
                              categories: [Dict[str, str]] = None) -> pd.DataFrame:
    """
    Create binary columns for all major categories.

    Args:
        df: DataFrame with the data
        min_frequency: Minimum occurrences for term inclusion
        categories: Dictionary mapping column names to mapping types
                   If None, uses default categories

    Returns:
        DataFrame with all binary columns added

    Example:
        df_full = create_all_binary_columns(df, min_frequency=10)
    """
    if categories is None:
        categories = {
            'platforms_detected': 'platform',
            'data_modalities_detected': 'modality',
            'task_types': 'task',
            'specific_models_mentioned': 'model'
        }

    df_result = df.copy()

    print("="*70)
    print("CREATING BINARY COLUMNS FOR ALL CATEGORIES")
    print("="*70)
    print()
    for col_name in categories:
        if col_name in df.columns:
            print(f"\nProcessing: {col_name}")
            print("-"*70)
            df_result = create_binary_columns(
                df_result,
                frequency_dict,
                col_name,
                min_frequency,
                None,
                categories
            )

    # Count new columns
    original_cols = len(df.columns)
    new_cols = len(df_result.columns) - original_cols

    print()
    print("="*70)
    print(f"✓ COMPLETE: Added {new_cols} binary columns")
    print(f"  Original columns: {original_cols}")
    print(f"  Total columns: {len(df_result.columns)}")
    print("="*70)

    return df_result

In [ ]:
def export_binary_columns(df: pd.DataFrame,
                         output_path: str,
                         include_original: bool = True) -> None:
    """
    Export dataframe with binary columns to CSV.

    Args:
        df: DataFrame with binary columns
        output_path: Path to save CSV
        include_original: Whether to include original columns
    """
    if include_original:
        df.to_csv(output_path, index=False)
    else:
        # Export only binary columns
        binary_cols = [col for col in df.columns if '_' in col and
                      any(prefix in col for prefix in ['task_', 'model_', 'modality_', 'platform_'])]
        df[binary_cols].to_csv(output_path, index=False)

    print(f"✓ Binary columns exported to: {output_path}")

## Part 2: Numeric Value Extraction Functions

In [ ]:
def extract_numbers_from_text(text: str, return_context: bool = False, return_single: bool = True, min_value: [float] = None,
                             require_context: bool = False) -> List[Dict]:
    """
    Extract numbers from text with proper handling of abbreviations and context.

    Args:
        text: Input text containing numbers
        return_context: If True, includes surrounding words for context

    Returns:
        List of dictionaries with extracted number information

    Examples:
        >>> extract_numbers_from_text('The dataset has 3.4M tweets')
        [{'raw': '3.4M', 'value': 3400000.0, 'unit': 'M', 'context': 'tweets'}]

        >>> extract_numbers_from_text('50,001 balanced text samples')
        [{'raw': '50,001', 'value': 50001.0, 'unit': None, 'context': 'samples'}]
    """
    results = []
    all_matches = []

    if pd.isna(text) or text == '':
        return np.nan

    # Pattern to match numbers with single-letter units (M, K, B)
    # Must not be part of a word (use word boundaries)
    #pattern_abbrev = r'(?<![a-zA-Z0-9])[~≈]?\s*(\d{1,3}(?:,\d{3})*(?:\.\d+)?)\s*([MKBmkb])(?![a-zA-Z])'
    pattern_abbrev = r'(?<![a-zA-Z0-9])[~≈∼]?\s*(\d+(?:[,\s]\d{3})*(?:\.\d+)?)\s*([KMBkmb])(?![a-zA-Z])'

    for match in re.finditer(pattern_abbrev, text):
        number_str = match.group(1)
        unit = match.group(2).upper()
        raw_text = match.group(0).strip()

        # Clean number: remove commas and spaces
        clean_num = number_str.replace(',', '').replace(' ', '')
        value = float(clean_num)

        # Apply multipliers
        if unit == 'K':
            value *= 1_000
        elif unit == 'M':
            value *= 1_000_000
        elif unit == 'B':
            value *= 1_000_000_000

        all_matches.append({
            'raw': raw_text,
            'value': value,
            'unit': unit,
            'start': match.start(),
            'end': match.end()
        })

    # PATTERN 2: Numbers with full word units (Million, Thousand, Billion)
    # Matches: 46 Million, 5 thousand, 2.5 Billion
    pattern_words = r'(?<![a-zA-Z0-9])[~≈∼]?\s*(\d+(?:[,\s]\d{3})*(?:\.\d+)?)\s+((?:M|m)illions?|(?:T|t)housands?|(?:B|b)illions?)\b'

    for match in re.finditer(pattern_words, text):
        # Check overlap with existing matches
        overlaps = any(match.start() >= m['start'] and match.start() < m['end'] for m in all_matches)
        if overlaps:
            continue

        number_str = match.group(1)
        unit_word = match.group(2).lower()
        raw_text = match.group(0).strip()

        # Clean number
        clean_num = number_str.replace(',', '').replace(' ', '')
        value = float(clean_num)

        # Apply multipliers
        if unit_word.startswith('t'):
            value *= 1_000
            unit = 'K'
        elif unit_word.startswith('m'):
            value *= 1_000_000
            unit = 'M'
        elif unit_word.startswith('b'):
            value *= 1_000_000_000
            unit = 'B'

        all_matches.append({
            'raw': raw_text,
            'value': value,
            'unit': unit,
            'start': match.start(),
            'end': match.end()
        })

    # PATTERN 3: Period-separated thousands (European format)
    # Matches: 7.786 → 7786, 123.456 → 123456, 1.234.567 → 1234567
    # Period with EXACTLY 3 digits = thousand separator
    # Period with 1-2 digits = decimal point (handled in pattern 6)
    # Must be EXACTLY 3 digits after each period, not followed by unit
    pattern_period = r'(?<![a-zA-Z0-9])[~≈∼]?\s*(\d{1,3}(?:\.\d{3})+)(?![a-zA-Z0-9])'

    for match in re.finditer(pattern_period, text):
        # Check overlap
        overlaps = any(match.start() >= m['start'] and match.start() < m['end'] for m in all_matches)
        if overlaps:
            continue

        number_str = match.group(1)
        raw_text = match.group(0).strip()

        # Remove periods (they're thousand separators, not decimals)
        clean_num = number_str.replace('.', '')
        value = float(clean_num)

        all_matches.append({
            'raw': raw_text,
            'value': value,
            'unit': None,
            'start': match.start(),
            'end': match.end()
        })

    # PATTERN 4: Space-separated thousands (European format)
    # Matches: 34 750 → 34750, 1 234 567 → 1234567
    # Must have at least one space between digit groups
    pattern_space = r'(?<![a-zA-Z0-9])[~≈∼]?\s*(\d{1,3}(?:\s\d{3})+)(?![a-zA-Z0-9.])'

    for match in re.finditer(pattern_space, text):
        # Check overlap
        overlaps = any(match.start() >= m['start'] and match.start() < m['end'] for m in all_matches)
        if overlaps:
            continue

        number_str = match.group(1)
        raw_text = match.group(0).strip()

        # Remove spaces and convert
        clean_num = number_str.replace(' ', '')
        value = float(clean_num)

        all_matches.append({
            'raw': raw_text,
            'value': value,
            'unit': None,
            'start': match.start(),
            'end': match.end()
        })

    # PATTERN 5: Regular numbers (comma-separated or plain)
    # Matches: 1,234,567 or 50,001 or plain numbers
    pattern_comma = r'(?<![a-zA-Z0-9])(\d{1,3}(?:,\d{3})+)(?![a-zA-Z0-9])'

    for match in re.finditer(pattern_comma, text):
        # Check overlap
        overlaps = any(match.start() >= m['start'] and match.start() < m['end'] for m in all_matches)
        if overlaps:
            continue

        number_str = match.group(1)
        raw_text = match.group(0).strip()

        # Remove commas
        value = float(number_str.replace(',', ''))

        all_matches.append({
            'raw': raw_text,
            'value': value,
            'unit': None,
            'start': match.start(),
            'end': match.end()
        })

    # PATTERN 6: Plain numbers (including decimals with 1-2 digits)
    # Matches: 239.5, 45.67, 123, but NOT 7.786 (that's handled as thousand separator)
    # Be careful not to match years or small irrelevant numbers
    pattern_plain = r'(?<![a-zA-Z0-9])(\d+(?:\.\d{1,2})?)(?![a-zA-Z0-9])'

    for match in re.finditer(pattern_plain, text):
        # Check overlap
        overlaps = any(match.start() >= m['start'] and match.start() < m['end'] for m in all_matches)
        if overlaps:
            continue

        number_str = match.group(1)
        raw_text = match.group(0).strip()
        value = float(number_str)

        all_matches.append({
            'raw': raw_text,
            'value': value,
            'unit': None,
            'start': match.start(),
            'end': match.end()
        })

    # Sort by position
    all_matches.sort(key=lambda x: x['start'])

    # Filter and add context
    dataset_keywords = [
        'tweet', 'post', 'user', 'image', 'video', 'sample', 'document',
        'comment', 'photo', 'entry', 'record', 'instance', 'example',
        'participant', 'respondent', 'observation', 'item', 'message',
        'review', 'rating', 'annotation', 'label', 'tag', 'geotagged'
    ]

    for match_info in all_matches:
        value = match_info['value']

        # Filter out years (1900-2099)
        #if 1900 <= value <= 2099 and not match_info['unit']:
        #    continue

        # Apply minimum value filter
        if min_value is not None and value < min_value:
            continue

        # Get context
        after_text = text[match_info['end']:match_info['end'] + 50].lower()
        before_text = text[max(0, match_info['start'] - 50):match_info['start']].lower()

        # For small numbers without units, check context
        if require_context and value < 1000 and not match_info['unit']:
            is_relevant = any(kw in after_text or kw in before_text for kw in dataset_keywords)
            if not is_relevant and len(text.strip().split()) > 3:
                continue

        # Extract context words
        context_words = []
        if return_context:
            context_text = before_text + ' ' + after_text
            words = re.findall(r'\b[a-z]+\b', context_text)
            context_words = [w for w in words if len(w) > 2]
        else:
            word_match = re.search(r'\b([a-z]+)\b', after_text)
            if word_match:
                context_words = [word_match.group(1)]

        result = {
            'raw': match_info['raw'],
            'value': match_info['value'],
            'unit': match_info['unit'],
            'context': ' '.join(context_words[:3]) if context_words else None,
            'position': match_info['start']
        }

        results.append(result)

    if results:
        if return_single:
            return max([k['value'] for k in results])
        else:
            return results
    else:
        return np.nan


In [ ]:
extract_numbers_from_text('2 papers', require_context=False)

In [ ]:
def convert_to_numeric(df, column_name, new_column_name=None):
    """
    Convert a text column to numeric values.
    """
    if new_column_name is None:
        new_column_name = f"{column_name}_numeric"
    
    df[new_column_name] = df[column_name].apply(extract_numbers_from_text)
    
    return df


print("✓ Numeric extraction functions loaded successfully")

## Part 3: Date Range Parsing Functions

In [ ]:
def parse_date_range(date_string: str):
    """
    Parse various date range formats and return start date, end date, and duration.

    Supported formats:
    - Single year: "2019" → Jan 1 - Dec 31, 2019
    - Year range: "2018-2020" → Jan 1, 2018 - Dec 31, 2020
    - Month-year range: "January 2019 to March 2020"
    - Within-year month range: "June to August in 2019"
    - Day-month range: "9 April-12 August 2020"
    - Specific dates: "April 9, 2019 to August 12, 2020"
    - Compound ranges: "9 April-12 August 2020 and 9 April-12 August 2019"

    Args:
        date_string: The date range string to parse

    Returns:
        Tuple of (start_date, end_date, duration_in_days)
        For compound ranges with "and", returns total combined duration
        Returns (None, None, None) if parsing fails

    Examples:
        >>> parse_date_range("2019")
        (datetime(2019, 1, 1), datetime(2019, 12, 31), 364)

        >>> parse_date_range("9 April-12 August 2020")
        (datetime(2020, 4, 9), datetime(2020, 8, 12), 125)

        >>> parse_date_range("9 April-12 August 2020 and 9 April-12 August 2019")
        (datetime(2019, 4, 9), datetime(2020, 8, 12), 250)  # Combined: 125 + 125 days
    """
    if not date_string or str(date_string).lower() in ['nan', 'none', '']:
        return None, None, None

    date_string = str(date_string).strip()

    try:
        # Handle compound ranges with "and" (e.g., "9 April-12 August 2020 and 9 April-12 August 2019")
        if ' and ' in date_string.lower():
            parts = re.split(r'\s+and\s+', date_string, flags=re.IGNORECASE)

            # Try to parse each part separately
            parsed_ranges = []
            for part in parts:
                start, end, duration = parse_date_range(part.strip())
                if start and end and duration is not None:
                    parsed_ranges.append((start, end, duration))

            if len(parsed_ranges) >= 2:
                # Return the full span: earliest start to latest end
                all_starts = [r[0] for r in parsed_ranges]
                all_ends = [r[1] for r in parsed_ranges]
                earliest_start = min(all_starts)
                latest_end = max(all_ends)

                # Calculate total duration as sum of individual periods
                total_duration = sum(r[2] for r in parsed_ranges)

                return earliest_start, latest_end, total_duration

            # If parsing compound range failed, continue to other formats

        # Format 1: Single year (e.g., "2019", "2020")
        if re.match(r'^\d{4}$', date_string):
            year = int(date_string)
            start_date = datetime(year, 1, 1)
            end_date = datetime(year, 12, 31)
            duration = (end_date - start_date).days
            return start_date, end_date, duration

        # Format 2: Year range (e.g., "2018-2020", "2018–2020")
        match = re.match(r'^(\d{4})[–\-](\d{4})$', date_string)
        if match:
            start_year = int(match.group(1))
            end_year = int(match.group(2))
            start_date = datetime(start_year, 1, 1)
            end_date = datetime(end_year, 12, 31)
            duration = (end_date - start_date).days
            return start_date, end_date, duration

        # Format 3: Day Month-Day Month Year (e.g., "9 April-12 August 2020")
        # This is the problematic format!
        match = re.match(r'^(\d{1,2})\s+(\w+)[–\-](\d{1,2})\s+(\w+)\s+(\d{4})$', date_string)
        if match:
            start_day = int(match.group(1))
            start_month_str = match.group(2)
            end_day = int(match.group(3))
            end_month_str = match.group(4)
            year = int(match.group(5))

            # Parse dates
            start_date = datetime.strptime(f"{start_day} {start_month_str} {year}", "%d %B %Y")
            end_date = datetime.strptime(f"{end_day} {end_month_str} {year}", "%d %B %Y")
            duration = (end_date - start_date).days
            return start_date, end_date, duration

        # Format 4: Month Year to Month Year (e.g., "January 2019 to March 2020")
        match = re.match(r'^(\w+)\s+(\d{4})\s+to\s+(\w+)\s+(\d{4})$', date_string, re.IGNORECASE)
        if match:
            start_month_str = match.group(1)
            start_year = int(match.group(2))
            end_month_str = match.group(3)
            end_year = int(match.group(4))

            start_date = datetime.strptime(f"{start_month_str} {start_year}", "%B %Y")
            end_date = datetime.strptime(f"{end_month_str} {end_year}", "%B %Y")
            # Get last day of end month
            last_day = monthrange(end_year, end_date.month)[1]
            end_date = datetime(end_date.year, end_date.month, last_day)

            duration = (end_date - start_date).days
            return start_date, end_date, duration

        # Format 5: Month to Month in Year (e.g., "June to August in 2019")
        match = re.match(r'^(\w+)\s+to\s+(\w+)\s+in\s+(\d{4})$', date_string, re.IGNORECASE)
        if match:
            start_month_str = match.group(1)
            end_month_str = match.group(2)
            year = int(match.group(3))

            start_date = datetime.strptime(f"{start_month_str} {year}", "%B %Y")
            end_date = datetime.strptime(f"{end_month_str} {year}", "%B %Y")
            # Get last day of end month
            if end_date.month == 12:
                last_day = 31
            else:
                last_day = monthrange(year, end_date.month)[1]
            end_date = datetime(year, end_date.month, last_day)

            duration = (end_date - start_date).days
            return start_date, end_date, duration

        # Format 6: Month Year-Month Year (e.g., "January 2019-March 2020")
        match = re.match(r'^(\w+)\s+(\d{4})[–\-](\w+)\s+(\d{4})$', date_string)
        if match:
            start_month_str = match.group(1)
            start_year = int(match.group(2))
            end_month_str = match.group(3)
            end_year = int(match.group(4))

            start_date = datetime.strptime(f"{start_month_str} {start_year}", "%B %Y")
            end_date = datetime.strptime(f"{end_month_str} {end_year}", "%B %Y")
            # Get last day of end month
            last_day = monthrange(end_year, end_date.month)[1]
            end_date = datetime(end_year, end_date.month, last_day)

            duration = (end_date - start_date).days
            return start_date, end_date, duration

        # Format 7: ISO format or standard dates
        # Try common date formats
        date_formats = [
            "%Y-%m-%d",
            "%d/%m/%Y",
            "%m/%d/%Y",
            "%B %d, %Y",
            "%d %B %Y",
        ]

        # Check if it's a date range with various separators
        for separator in [' to ', '-', '–', ' - ', ' – ']:
            if separator in date_string:
                parts = date_string.split(separator)
                if len(parts) == 2:
                    start_str, end_str = parts[0].strip(), parts[1].strip()

                    # Try to parse both parts
                    start_date, end_date = None, None
                    for fmt in date_formats:
                        try:
                            start_date = datetime.strptime(start_str, fmt)
                            break
                        except:
                            continue

                    for fmt in date_formats:
                        try:
                            end_date = datetime.strptime(end_str, fmt)
                            break
                        except:
                            continue

                    if start_date and end_date:
                        duration = (end_date - start_date).days
                        return start_date, end_date, duration

        # If nothing worked, return None
        return None, None, None

    except Exception as e:
        print(f"Error parsing date '{date_string}': {e}")
        return None, None, None

def parse_date_columns(df, date_column='dataset_time_coverage'):
    """
    Parse date range column and create start_date, end_date, and duration columns.
    """
    results = df[date_column].apply(parse_date_range)
    
    df['start_date'] = results.apply(lambda x: x[0] if x else None)
    df['end_date'] = results.apply(lambda x: x[1] if x else None)
    df['duration_days'] = results.apply(lambda x: x[2] if x else np.nan)
    
    return df


print("✓ Date parsing functions loaded successfully")

## Part 4: Load and Process Dataset 1

In [ ]:
# Load the first dataset
df1 = pd.read_csv('data/term_mapping/Gemini_2_flash_all/complete_dataset.csv')
df1['all_models'] = df1['model_families'].fillna('') + '; ' + df1['specific_models_mentioned'].fillna('')
df1['all_heritage'] = df1['heritage_terms_detected'].fillna('') + '; ' + df1['heritage_management_actions'].fillna('')

print(f"Dataset 1 loaded: {df1.shape[0]} rows, {df1.shape[1]} columns")
print(f"\nFirst few rows:")
df1.head(3)

In [ ]:
df_meta = pd.read_csv('data/topic_modeling/meta_data.csv')
df_meta['filename'] = df_meta['File_Name'] + '.txt'

print(f"Dataset 1 loaded: {df_meta.shape[0]} rows, {df_meta.shape[1]} columns")
print(f"\nFirst few rows:")
df_meta.head(3)

### Task 1.1: Extract Frequencies for Platforms, Modalities, Tasks, and Models

In [ ]:
freq_dicts_df1 = analyze_dataset(df1)
for col_name, freq_dict in freq_dicts_df1.items():
        print_frequency_summary(freq_dict, col_name, top_n=25)

In [ ]:
export_frequencies_to_json(freq_dicts_df1, 'data/term_mapping/Gemini_2_flash_all/consolidated_frequencies.json')
print("✓ Results exported to consolidated_frequencies.json")

In [ ]:
df1_binary = create_all_binary_columns(df1, freq_dicts_df1)

In [ ]:
df1_binary

In [ ]:
export_binary_columns(df1_binary, 'data/term_mapping/Gemini_2_flash_all/dataset_with_binary_columns.csv')

### Task 1.2: Convert Dataset Size Metrics to Numeric

In [ ]:
# Convert dataset_total_posts
df1 = convert_to_numeric(df1, 'dataset_total_posts', 'total_posts_numeric')

# Convert dataset_total_users
df1 = convert_to_numeric(df1, 'dataset_total_users', 'total_users_numeric')

print("\n" + "="*60)
print("NUMERIC CONVERSION RESULTS")
print("="*60)

print("\nDataset Total Posts:")
print(f"  Non-null values: {df1['total_posts_numeric'].notna().sum()}")
print(f"  Min: {df1['total_posts_numeric'].min():,.0f}")
print(f"  Max: {df1['total_posts_numeric'].max():,.0f}")
print(f"  Mean: {df1['total_posts_numeric'].mean():,.0f}")
print(f"  Median: {df1['total_posts_numeric'].median():,.0f}")

print("\nDataset Total Users:")
print(f"  Non-null values: {df1['total_users_numeric'].notna().sum()}")
if df1['total_users_numeric'].notna().any():
    print(f"  Min: {df1['total_users_numeric'].min():,.0f}")
    print(f"  Max: {df1['total_users_numeric'].max():,.0f}")
    print(f"  Mean: {df1['total_users_numeric'].mean():,.0f}")
    print(f"  Median: {df1['total_users_numeric'].median():,.0f}")

# Show sample conversions
print("\nSample Conversions:")
sample_df = df1[['dataset_total_posts', 'total_posts_numeric', 'dataset_total_users', 'total_users_numeric']].head(10)
sample_df

### Task 1.3: Parse Date Ranges and Calculate Duration

In [ ]:
# Parse date ranges
df1 = parse_date_columns(df1, 'dataset_time_coverage')

print("\n" + "="*60)
print("DATE RANGE PARSING RESULTS")
print("="*60)

print(f"\nTotal records with date information: {df1['start_date'].notna().sum()}")
print(f"Earliest start date: {df1['start_date'].min()}")
print(f"Latest end date: {df1['end_date'].max()}")
print(f"\nDuration statistics (in days):")
print(f"  Min: {df1['duration_days'].min():.0f}")
print(f"  Max: {df1['duration_days'].max():.0f}")
print(f"  Mean: {df1['duration_days'].mean():.0f}")
print(f"  Median: {df1['duration_days'].median():.0f}")

# Show sample date conversions
print("\nSample Date Conversions:")
date_sample = df1[['dataset_time_coverage', 'start_date', 'end_date', 'duration_days']].dropna(subset=['start_date']).head(10)
date_sample

In [ ]:
df1.sort_values('total_posts_numeric', inplace=False, ascending=False)

In [ ]:
export_binary_columns(df1, 'data/term_mapping/Gemini_2_flash_all/dataset_with_numerics.csv')

### Task 1.4: Temporal Mapping

In [ ]:
def extract_year(df: pd.DataFrame, year_column: str = 'year') -> pd.DataFrame:
    """
    Extract year as integer from year column.

    Args:
        df: DataFrame
        year_column: Name of year column

    Returns:
        DataFrame with 'year_int' column
    """
    df = df.copy()

    # Convert year to integer
    df['year_int'] = pd.to_numeric(df[year_column], errors='coerce')

    # Remove rows with invalid years
    valid_years = df['year_int'].notna()
    if not valid_years.all():
        print(f"Warning: {(~valid_years).sum()} rows have invalid years and will be excluded")

    return df


def create_year_intervals(df: pd.DataFrame,
                         interval: int = 1,
                         merge_below: int = None,
                         merge_above: int = None,
                         year_column: str = 'year_int') -> pd.DataFrame:
    """
    Create year interval groups (e.g., yearly, bi-yearly).

    Args:
        df: DataFrame with year_int column
        interval: Interval size (1=yearly, 2=bi-yearly, 5=five-year, etc.)
        year_column: Name of year column

    Returns:
        DataFrame with 'year_group' column
    """
    df = df.copy()

    if year_column not in df.columns:
        df = extract_year(df)
        year_column = 'year_int'

    # Create intervals
    df['year_group'] = (df[year_column] // interval) * interval
    if merge_below is not None:
        df['year_group'] = df['year_group'].apply(lambda x: merge_below if x < merge_below else x)
    if merge_above is not None:
        max_year = sorted(df['year_group'].unique())[-2]
        df['year_group'] = df['year_group'].apply(lambda x: max_year if x > merge_above else x)

    # Create readable labels
    if interval == 1:
        df['year_label'] = df['year_group'].astype(int).astype(str)
    else:
        df['year_label'] = df['year_group'].apply(
            lambda x: f"{int(x)}-{int(x + interval - 1)}" if pd.notna(x) else None
        )
    if merge_below is not None:
        df['year_label'] = df['year_label'].apply(lambda x: 'before '+ str(merge_below) if x.startswith(str(merge_below)) else x)
    if merge_above is not None:
        df['year_label'] = df['year_label'].apply(lambda x: 'after '+ str(max_year) if x.startswith(str(max_year)) else x)

    return df


def get_temporal_counts(df: pd.DataFrame,
                        interval: int = 1,
                        merge_below: int = None,
                         merge_above: int = None,
                         year_column: str = 'year_int') -> pd.DataFrame:
    """
    Get counts of terms over time intervals.

    Args:
        df: DataFrame with data

    Returns:
        DataFrame with year_label as index, terms as columns, counts as values
    """
    # Prepare data with year groups

    df_prep = extract_year(df, year_column= 'Publication Year')
    df_prep = create_year_intervals(df_prep, interval, merge_below, merge_above, year_column)
    df_prep = df_prep[df_prep['year_label'].notna()].copy()

    binary_cols = [col for col in df_prep.columns if '_' in col and any(col.startswith(prefix)  for prefix in ['task_', 'model_', 'modality_', 'platform_'])]

    # Create temporal count matrix by grouping and summing binary columns
    temporal_df = df_prep.groupby('year_label')[binary_cols].sum()

    return temporal_df

In [ ]:
def get_temporal_dfs(df, meta_df):
    df_prep = df.merge(meta_df[['Publication Year', 'filename']], left_on='filename', right_on='filename', how='inner')
    df_prep = extract_year(df_prep, year_column= 'Publication Year')
    df_prep = create_year_intervals(df_prep, interval=2, merge_below=2013, merge_above=2025)
    temporal_df = get_temporal_counts(df.merge(meta_df[['Publication Year', 'filename']], left_on='filename', right_on='filename', how='inner'), interval=2, merge_below=2013, merge_above=2025).iloc[:, 4:].reindex(df_prep['year_label'].value_counts(ascending=True).index)

    return df_prep, temporal_df

In [ ]:
df_prep = df1_binary.merge(df_meta[['Publication Year', 'filename']], left_on='filename', right_on='filename', how='inner')
df_prep = extract_year(df_prep, year_column= 'Publication Year')
df_prep = create_year_intervals(df_prep, interval=2, merge_below=2013, merge_above=2025)
df_prep

In [ ]:
df_prep['year_int'].hist(bins=17)

In [ ]:
df_prep['year_label'].value_counts(ascending=True)

In [ ]:
df_prep['year_label'].value_counts(ascending=True).to_frame().T.plot(kind='barh', stacked=True, colormap='Grays', legend=False, figsize=(15,1))

In [ ]:
temporal_df = get_temporal_counts(df1_binary.merge(df_meta[['Publication Year', 'filename']], left_on='filename', right_on='filename', how='inner'), interval=2, merge_below=2013, merge_above=2025).iloc[:, 4:].reindex(df_prep['year_label'].value_counts(ascending=True).index)

### Visualizations for Dataset 1

In [ ]:
def plot_temporal_trends(temporal_df: pd.DataFrame,
                        column_name: str,
                        interval: int = 1,
                        figsize: Tuple[int, int] = (14, 8),
                        title: [str] = None,
                        output_path: [str] = None,
                        stacked: bool = True,
                        colormap: str = 'Blues',
                        reference_on: bool = False,
                        ref_df: pd.DataFrame = None) -> None:
    """
    Create stacked bar plot showing temporal trends.

    Args:
        df: DataFrame with data (can have binary columns for faster processing)
        column_name: Column to analyze
        mapping_type: Type of mapping
        interval: Year interval (1=yearly, 2=bi-yearly, etc.)
        top_n: Show top N terms
        min_frequency: Minimum total frequency
        figsize: Figure size
        title: Plot title (auto-generated if None)
        output_path: Path to save figure (if provided)
        stacked: Whether to stack bars
        colormap: Matplotlib colormap name
        use_binary_columns: If True and binary columns exist, use them (much faster!)

    Example:
        plot_temporal_trends(df, 'task_types', 'task', interval=2, top_n=15)
    """
    # Get temporal counts
    # Create figure

    fig = plt.figure(figsize=figsize)
    if reference_on:
        gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[1, 20], hspace=0.001)
        ax = fig.add_subplot(gs[1])
        ax_ref  = fig.add_subplot(gs[0])
        ax_ref.set_frame_on(False)
        ax_ref.tick_params(axis="x", which="both", bottom=False, top=False, labelbottom=False)
        ax_ref.tick_params(axis="y", left=False, right=False)
    else:
        ax = fig.add_subplot(111)

    cols = temporal_df.columns
    rename_cols = {col: col.replace(column_name + '_', '') for col in cols}
    temporal_df = temporal_df.rename(columns=rename_cols)

    categories = {
        'platform': "Social Media Platforms",
        'model': "AI Models",
        'task': "Computational Tasks",
        'modality': "Data Modality",
    }

    title_cat = categories[column_name]

    top_n = len(cols)

    # Plot stacked bar chart
    if stacked:
        temporal_df.T.plot(kind='barh', stacked=True, ax=ax,
                        colormap=colormap, width=0.8)
    else:
        temporal_df.T.plot(kind='barh', ax=ax,
                        colormap=colormap, width=0.8)

    if reference_on:
        ref_df['year_label'].value_counts(ascending=True).to_frame().rename(columns = {'count': "Ref. (Num Papers)"}).T.plot(kind='barh', stacked=True, colormap=colormap, legend=False, ax=ax_ref)

    # Customize plot
    if title is None:
        interval_name = f"{interval}-year" if interval > 1 else "Yearly"
        title = f'{interval_name} Trends: Top {top_n} {title_cat}'

    if reference_on:
        ax_ref.set_title(title, fontsize=16, fontweight='bold', pad=5)

    else:
        ax.set_title(title, fontsize=16, fontweight='bold', pad=20)

    ax.set_xlabel('Number of Papers', fontsize=14, fontweight='bold')
    #ax.set_ylabel('Number of Papers', fontsize=12, fontweight='bold')
    ax.invert_yaxis()

    # Rotate x-axis labels
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=12)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=12)

    # Legend
    ax.legend(title='Year Group',
             loc='lower right', fontsize=11, title_fontsize=12)

    # Grid
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.set_axisbelow(True)

    plt.tight_layout()

    # Save if path provided
    if output_path:
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        print(f"✓ Plot saved to: {output_path}")

    plt.show()

In [ ]:
color_cats = {
    'platform': "Blues",
    'modality': "Greens",
    'model': "Oranges",
    'task': "Reds",

}

df_prep, temporal_df = get_temporal_dfs(df1_binary, df_meta)

for k, v in color_cats.items():
    plot_temporal_trends(temporal_df[[col for col in temporal_df if col.startswith(k)]], k, colormap = v, reference_on=True, ref_df=df_prep,
                         output_path = 'data/term_mapping/Gemini_2_flash_all/{}.jpeg'.format(k))

In [ ]:
# Visualization 1: Top Platforms
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Platforms
if 'platforms_detected' in freq_dicts_df1:
    top_platforms = dict(list(freq_dicts_df1['platforms_detected'].items())[:21])
    axes[0, 0].barh(list(top_platforms.keys()), list(top_platforms.values()), color='steelblue')
    axes[0, 0].set_xlabel('Frequency')
    axes[0, 0].set_title('Top 10 Social Media Platforms', fontsize=14, fontweight='bold')
    axes[0, 0].invert_yaxis()

# Data Modalities
if 'data_modalities_detected' in freq_dicts_df1:
    top_modalities = dict(list(freq_dicts_df1['data_modalities_detected'].items())[:17])
    axes[0, 1].barh(list(top_modalities.keys()), list(top_modalities.values()), color='forestgreen')
    axes[0, 1].set_xlabel('Frequency')
    axes[0, 1].set_title('Top 10 Data Modalities', fontsize=14, fontweight='bold')
    axes[0, 1].invert_yaxis()

# Task Types
if 'task_types' in freq_dicts_df1:
    top_tasks = dict(list(freq_dicts_df1['task_types'].items())[:13])
    axes[1, 0].barh(list(top_tasks.keys()), list(top_tasks.values()), color='darkorange')
    axes[1, 0].set_xlabel('Frequency')
    axes[1, 0].set_title('Top 10 AI Task Types', fontsize=14, fontweight='bold')
    axes[1, 0].invert_yaxis()

# Models
if 'specific_models_mentioned' in freq_dicts_df1:
    top_models = dict(list(freq_dicts_df1['specific_models_mentioned'].items())[:16])
    axes[1, 1].barh(list(top_models.keys()), list(top_models.values()), color='crimson')
    axes[1, 1].set_xlabel('Frequency')
    axes[1, 1].set_title('Top 10 AI Models', fontsize=14, fontweight='bold')
    axes[1, 1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: Dataset size distributions
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Posts distribution
posts_data = df1['total_posts_numeric'].dropna()
if len(posts_data) > 0:
    axes[0].hist(np.log10(posts_data + 1), bins=12, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Log10(Total Posts + 1)')
    axes[0].set_xlim((0,9))
    axes[0].set_ylim((0,120))
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Dataset Sizes (Posts)', fontsize=14, fontweight='bold')
    axes[0].grid(axis='y', alpha=0.3)

# Duration distribution
duration_data = df1['duration_days'].dropna()
if len(duration_data) > 0:
    axes[1].hist(duration_data, bins=30, color='forestgreen', edgecolor='black', alpha=0.7)
    axes[1].set_xlabel('Duration (days)')
    axes[1].set_ylabel('Frequency')
    axes[1].set_ylim((0,22))
    axes[1].set_title('Distribution of Study Durations', fontsize=14, fontweight='bold')
    axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Part 5: Load and Process Dataset 2 (for comparison)

**Instructions:** 
- Update the file path below to your second dataset
- The dataset should have the same column structure
- Run the cells below to perform the same analysis and comparison

In [ ]:
# MODIFY THIS PATH to your second dataset
DATASET_2_PATH = 'data/term_mapping/Gemini_2_5_pro/complete_dataset.csv'

# Check if second dataset exists
import os
if os.path.exists(DATASET_2_PATH):
    df2 = pd.read_csv(DATASET_2_PATH)
    print(f"Dataset 2 loaded: {df2.shape[0]} rows, {df2.shape[1]} columns")

    df2['all_models'] = df2['model_families'] + '; ' + df2['specific_models_mentioned']

    # Perform all transformations on dataset 2
    print("\nProcessing Dataset 2...")

In [ ]:
results_2 = analyze_dataset(df2)
for col_name, freq_dict in results_2.items():
    print_frequency_summary(freq_dict, col_name, top_n=15)

In [ ]:
export_frequencies_to_json(results_2, 'data/term_mapping/Gemini_2_5_pro/consolidated_frequencies.json')
print("✓ Results exported to consolidated_frequencies.json")

In [ ]:
df2_binary = create_all_binary_columns(df2, results_2)

In [ ]:
export_binary_columns(df2_binary, 'data/term_mapping/Gemini_2_5_pro/dataset_with_binary_columns.csv')

In [ ]:
df2_prep, temporal2_df = get_temporal_dfs(df2_binary, df_meta)

for k, v in color_cats.items():
    if not k=='platform':
        plot_temporal_trends(temporal2_df[[col for col in temporal2_df if col.startswith(k)]], k, colormap = v, reference_on=True, ref_df=df2_prep,
                         output_path = 'data/term_mapping/Gemini_2_5_pro/{}.jpeg'.format(k))

In [ ]:
temporal2_df

In [ ]:
# Visualization 1: Top Platforms
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Platforms
if 'platforms_detected' in results_1:
    top_platforms = dict(list(results_1['platforms_detected'].items())[:15])
    axes[0, 0].barh(list(top_platforms.keys()), list(top_platforms.values()), color='steelblue')
    axes[0, 0].set_xlabel('Frequency')
    axes[0, 0].set_title('Top 10 Social Media Platforms', fontsize=14, fontweight='bold')
    axes[0, 0].invert_yaxis()

# Data Modalities
if 'data_modalities_detected' in results_2:
    top_modalities = dict(list(results_2['data_modalities_detected'].items())[:20])
    axes[0, 1].barh(list(top_modalities.keys()), list(top_modalities.values()), color='forestgreen')
    axes[0, 1].set_xlabel('Frequency')
    axes[0, 1].set_title('Top 10 Data Modalities', fontsize=14, fontweight='bold')
    axes[0, 1].invert_yaxis()

# Task Types
if 'task_types' in results_2:
    top_tasks = dict(list(results_2['task_types'].items())[:22])
    axes[1, 0].barh(list(top_tasks.keys()), list(top_tasks.values()), color='darkorange')
    axes[1, 0].set_xlabel('Frequency')
    axes[1, 0].set_title('Top 10 AI Task Types', fontsize=14, fontweight='bold')
    axes[1, 0].invert_yaxis()

# Models
if 'specific_models_mentioned' in results_2:
    top_models = dict(list(results_2['specific_models_mentioned'].items())[:31])
    axes[1, 1].barh(list(top_models.keys()), list(top_models.values()), color='crimson')
    axes[1, 1].set_xlabel('Frequency')
    axes[1, 1].set_title('Top 10 AI Models', fontsize=14, fontweight='bold')
    axes[1, 1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Numeric conversions
df2 = convert_to_numeric(df2, 'dataset_total_posts', 'total_posts_numeric')
df2 = convert_to_numeric(df2, 'dataset_total_users', 'total_users_numeric')

# Task 3: Date parsing
df2 = parse_date_columns(df2, 'dataset_time_coverage')

print("✓ Dataset 2 processing complete")

In [ ]:
df2[df2['total_posts_numeric']>10000000]

In [ ]:
print("\n" + "="*60)
print("NUMERIC CONVERSION RESULTS")
print("="*60)

print("\nDataset Total Posts:")
print(f"  Non-null values: {df2['total_posts_numeric'].notna().sum()}")
print(f"  Min: {df2['total_posts_numeric'].min():,.0f}")
print(f"  Max: {df2['total_posts_numeric'].max():,.0f}")
print(f"  Mean: {df2['total_posts_numeric'].mean():,.0f}")
print(f"  Median: {df2['total_posts_numeric'].median():,.0f}")

print("\nDataset Total Users:")
print(f"  Non-null values: {df2['total_users_numeric'].notna().sum()}")
if df1['total_users_numeric'].notna().any():
    print(f"  Min: {df2['total_users_numeric'].min():,.0f}")
    print(f"  Max: {df2['total_users_numeric'].max():,.0f}")
    print(f"  Mean: {df2['total_users_numeric'].mean():,.0f}")
    print(f"  Median: {df2['total_users_numeric'].median():,.0f}")

# Show sample conversions
print("\nSample Conversions:")
sample_df = df2[['dataset_total_posts', 'total_posts_numeric', 'dataset_total_users', 'total_users_numeric']].head(10)
sample_df

In [ ]:
df2.sort_values('total_posts_numeric', inplace=False, ascending=False)

In [ ]:
df2.sort_values('total_posts_numeric', inplace=False)

In [ ]:
df2[df2['filename'].str.startswith('_Bai')]

In [ ]:
df2['total_posts_numeric'].describe()

In [ ]:
df2['end_date'].describe()

In [ ]:
export_binary_columns(df2, 'data/term_mapping/Gemini_2_5_pro/dataset_with_numerics.csv')

In [ ]:
# Visualization 2: Dataset size distributions
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Posts distribution
posts_data = df2['total_posts_numeric'].dropna()
if len(posts_data) > 0:
    axes[0].hist(np.log10(posts_data + 1), bins=12, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Log10(Total Posts + 1)', fontsize=12)
    axes[0].set_xlim((0,9))
    axes[0].set_ylim((0,100))
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Dataset Sizes (Posts)', fontsize=14, fontweight='bold')
    axes[0].grid(axis='y', alpha=0.3)

# Duration distribution
duration_data = df2['duration_days'].dropna()
if len(duration_data) > 0:
    axes[1].hist(duration_data, bins=30, color='forestgreen', edgecolor='black', alpha=0.7)
    axes[1].set_xlabel('Duration (days)')
    axes[1].set_ylabel('Frequency')
    axes[1].set_ylim((0,22))
    axes[1].set_title('Distribution of Study Durations', fontsize=14, fontweight='bold')
    axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: Dataset size distributions
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Posts distribution
posts_data = df1['total_posts_numeric'].dropna()
if len(posts_data) > 0:
    axes[0].hist(np.log10(posts_data + 1), bins=30, range=(0, 12), color='steelblue', edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Log10(Total Posts + 1)', fontsize=12)
    axes[0].set_xlim((0,11))
    axes[0].set_ylim((0,60))
    axes[0].set_ylabel('Frequency', fontsize=12)
    axes[0].set_title('Distribution of Dataset Sizes\n(Extracted from gemini-flash-2.0)', fontsize=15, fontweight='bold')
    axes[0].grid(axis='y', alpha=0.3)

# Duration distribution
posts_data = df2['total_posts_numeric'].dropna()
if len(posts_data) > 0:
    axes[1].hist(np.log10(posts_data + 1), bins=30, range=(0, 12), color='steelblue', edgecolor='black', alpha=0.7)
    axes[1].set_xlabel('Log10(Total Posts + 1)', fontsize=12)
    axes[1].set_xlim((0,11))
    axes[1].set_ylim((0,60))
    axes[1].set_ylabel('Frequency', fontsize=12)
    axes[1].set_title('Distribution of Dataset Sizes\n(Extracted from gemini-pro-2.5)', fontsize=15, fontweight='bold')
    axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Part 6: Cross-Dataset Comparison

In [ ]:
def compare_frequency_dicts(dict1, dict2, name1="Dataset 1", name2="Dataset 2", top_n=20):
    """
    Compare two frequency dictionaries and identify overlaps and differences.
    """
    print(f"\n{'='*80}")
    print(f"COMPARISON: {name1.upper()} vs {name2.upper()}")
    print(f"{'='*80}")
    
    keys1 = set(dict1.keys())
    keys2 = set(dict2.keys())
    
    common_keys = keys1 & keys2
    only_in_1 = keys1 - keys2
    only_in_2 = keys2 - keys1
    
    print(f"\nUnique items in {name1}: {len(keys1)}")
    print(f"Unique items in {name2}: {len(keys2)}")
    print(f"Common items: {len(common_keys)} ({len(common_keys)/max(len(keys1), len(keys2))*100:.1f}% overlap)")
    print(f"Only in {name1}: {len(only_in_1)}")
    print(f"Only in {name2}: {len(only_in_2)}")
    
    # Show top items in common
    if common_keys:
        print(f"\nTop {min(10, len(common_keys))} items in both datasets:")
        common_items = [(k, dict1[k], dict2[k]) for k in common_keys]
        common_items.sort(key=lambda x: x[1] + x[2], reverse=True)
        
        for i, (item, count1, count2) in enumerate(common_items[:10], 1):
            print(f"  {i:2d}. {item:30s} | {name1}: {count1:4d} | {name2}: {count2:4d}")
    
    # Show items only in dataset 1
    if only_in_1:
        print(f"\nTop {min(5, len(only_in_1))} items only in {name1}:")
        only_1_items = [(k, dict1[k]) for k in only_in_1]
        only_1_items.sort(key=lambda x: x[1], reverse=True)
        
        for i, (item, count) in enumerate(only_1_items[:5], 1):
            print(f"  {i}. {item:30s} : {count:4d}")
    
    # Show items only in dataset 2
    if only_in_2:
        print(f"\nTop {min(5, len(only_in_2))} items only in {name2}:")
        only_2_items = [(k, dict2[k]) for k in only_in_2]
        only_2_items.sort(key=lambda x: x[1], reverse=True)
        
        for i, (item, count) in enumerate(only_2_items[:5], 1):
            print(f"  {i}. {item:30s} : {count:4d}")
    
    return {
        'common': len(common_keys),
        'only_1': len(only_in_1),
        'only_2': len(only_in_2),
        'overlap_pct': len(common_keys)/max(len(keys1), len(keys2))*100 if max(len(keys1), len(keys2)) > 0 else 0
    }


if df2 is not None:
    comparison_results = {}
    
    for col in columns_to_analyze:
        if col in results_1 and col in results_2:
            results = compare_frequency_dicts(
                freq_dicts_df1[col],
                results_2[col],
                name1="Dataset 1",
                name2="Dataset 2"
            )
            comparison_results[col] = results
else:
    print("Dataset 2 not loaded. Skipping comparison.")

In [ ]:
if df2 is not None:
    # Compare numeric statistics
    print("\n" + "="*80)
    print("NUMERIC STATISTICS COMPARISON")
    print("="*80)
    
    numeric_cols = ['total_posts_numeric', 'total_users_numeric', 'duration_days']
    
    comparison_df = pd.DataFrame({
        'Metric': [],
        'Dataset 1 Mean': [],
        'Dataset 2 Mean': [],
        'Dataset 1 Median': [],
        'Dataset 2 Median': [],
        'Dataset 1 Non-null': [],
        'Dataset 2 Non-null': []
    })
    
    for col in numeric_cols:
        if col in df1.columns and col in df2.columns:
            comparison_df = pd.concat([comparison_df, pd.DataFrame({
                'Metric': [col],
                'Dataset 1 Mean': [df1[col].mean()],
                'Dataset 2 Mean': [df2[col].mean()],
                'Dataset 1 Median': [df1[col].median()],
                'Dataset 2 Median': [df2[col].median()],
                'Dataset 1 Non-null': [df1[col].notna().sum()],
                'Dataset 2 Non-null': [df2[col].notna().sum()]
            })], ignore_index=True)
    
    display(comparison_df)
else:
    print("Dataset 2 not loaded. Skipping numeric comparison.")

In [ ]:
results_2.keys()

In [ ]:
if df2 is not None:
    freq_dicts_df2 = results_2
    # Visualization: Side-by-side comparison
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    
    # Compare platforms
    if 'data_modalities_detected' in freq_dicts_df1 and 'data_modalities_detected' in freq_dicts_df2:
        all_platforms = set(list(freq_dicts_df1['data_modalities_detected'].keys())[:15] +
                           list(freq_dicts_df2['data_modalities_detected'].keys())[:15])
        platforms = list(all_platforms)[:15]
        
        counts1 = [freq_dicts_df1['data_modalities_detected'].get(p, 0) for p in platforms]
        counts2 = [freq_dicts_df2['data_modalities_detected'].get(p, 0) for p in platforms]
        
        x = np.arange(len(platforms))
        width = 0.35
        
        axes[0, 0].barh(x - width/2, counts1, width, label='Dataset 1', color='steelblue')
        axes[0, 0].barh(x + width/2, counts2, width, label='Dataset 2', color='coral')
        axes[0, 0].set_yticks(x)
        axes[0, 0].set_yticklabels(platforms)
        axes[0, 0].set_xlabel('Frequency')
        axes[0, 0].set_title('Modality Comparison', fontsize=14, fontweight='bold')
        axes[0, 0].legend()
        axes[0, 0].invert_yaxis()
    
    # Compare tasks
    if 'task_types' in freq_dicts_df1 and 'task_types' in freq_dicts_df2:
        all_tasks = set(list(freq_dicts_df1['task_types'].keys())[:15] +
                       list(freq_dicts_df2['task_types'].keys())[:15])
        tasks = list(all_tasks)[:15]
        
        counts1 = [freq_dicts_df1['task_types'].get(t, 0) for t in tasks]
        counts2 = [freq_dicts_df2['task_types'].get(t, 0) for t in tasks]
        
        x = np.arange(len(tasks))
        
        axes[0, 1].barh(x - width/2, counts1, width, label='Dataset 1', color='forestgreen')
        axes[0, 1].barh(x + width/2, counts2, width, label='Dataset 2', color='gold')
        axes[0, 1].set_yticks(x)
        axes[0, 1].set_yticklabels(tasks)
        axes[0, 1].set_xlabel('Frequency')
        axes[0, 1].set_title('Task Types Comparison', fontsize=14, fontweight='bold')
        axes[0, 1].legend()
        axes[0, 1].invert_yaxis()
    
    # Compare dataset sizes
    axes[1, 0].hist([np.log10(df1['total_posts_numeric'].dropna() + 1), 
                     np.log10(df2['total_posts_numeric'].dropna() + 1)],
                    bins=20, label=['Dataset 1', 'Dataset 2'], 
                    color=['steelblue', 'coral'], alpha=0.6)
    axes[1, 0].set_xlabel('Log10(Total Posts + 1)')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Dataset Size Comparison', fontsize=14, fontweight='bold')
    axes[1, 0].legend()
    axes[1, 0].grid(axis='y', alpha=0.3)
    
    # Compare durations
    axes[1, 1].hist([df1['duration_days'].dropna(), df2['duration_days'].dropna()],
                    bins=20, label=['Dataset 1', 'Dataset 2'],
                    color=['forestgreen', 'gold'], alpha=0.6)
    axes[1, 1].set_xlabel('Duration (days)')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title('Study Duration Comparison', fontsize=14, fontweight='bold')
    axes[1, 1].legend()
    axes[1, 1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("Dataset 2 not loaded. Skipping visualization comparison.")

## Part 7: Export Results

In [ ]:
# Export processed Dataset 1
output_file_1 = 'outputs/dataset1_processed.csv'
df1.to_csv(output_file_1, index=False)
print(f"✓ Dataset 1 (processed) saved to: {output_file_1}")

# Export frequency dictionaries as JSON
import json

freq_output_1 = 'outputs/dataset1_frequencies.json'
with open(freq_output_1, 'w') as f:
    json.dump(freq_dicts_df1, f, indent=2)
print(f"✓ Dataset 1 frequencies saved to: {freq_output_1}")

if df2 is not None:
    output_file_2 = 'outputs/dataset2_processed.csv'
    df2.to_csv(output_file_2, index=False)
    print(f"✓ Dataset 2 (processed) saved to: {output_file_2}")
    
    freq_output_2 = 'outputs/dataset2_frequencies.json'
    with open(freq_output_2, 'w') as f:
        json.dump(freq_dicts_df2, f, indent=2)
    print(f"✓ Dataset 2 frequencies saved to: {freq_output_2}")
    
    # Export comparison results
    comparison_output = 'outputs/dataset_comparison.json'
    with open(comparison_output, 'w') as f:
        json.dump(comparison_results, f, indent=2)
    print(f"✓ Comparison results saved to: {comparison_output}")

print("\n✓ All results exported successfully!")

## Summary

This notebook has completed the following tasks:

1. **Frequency Analysis**: Extracted and normalized frequencies for:
   - Social media platforms
   - Data modalities
   - AI task types
   - Model families
   - Specific models mentioned

2. **Numeric Conversion**: Transformed text descriptions to numeric values for:
   - Total posts
   - Total users

3. **Date Parsing**: Extracted start dates, end dates, and calculated durations from time coverage descriptions

4. **Cross-Dataset Comparison**: Compared consistency between two datasets (if second dataset provided)

All processed data and frequency dictionaries have been exported for further analysis.